# 🤖 What is an AI Agent?

An **AI Agent** is a software system that can **perceive its environment, reason about what to do, take actions, and learn or adapt** to achieve a specific goal.

A simple way to remember it:

> **AI Agent = Goal + Reasoning + Tools + Actions + Feedback**

### 🔄 How an AI Agent Works

1. **Perceive** 👀 — Collects information from the user, files, APIs, sensors, websites, databases, etc.
2. **Reason** 🧠 — Understands the goal, analyzes the information, and decides what needs to be done.
3. **Plan** 📝 — Breaks a complex goal into smaller steps.
4. **Act** ⚙️ — Uses tools such as search, code execution, APIs, databases, or applications.
5. **Observe & Feedback** 🔍 — Checks the result of its actions and adjusts its approach when necessary.
6. **Complete the Goal** 🎯 — Continues the process until the task is completed or it determines that it cannot proceed.

An AI agent is therefore more than just a chatbot: **a chatbot mainly generates responses, while an agent can autonomously decide and execute multiple actions to accomplish a goal.**

## Now we will pickup the ```Price is Right``` Project and put it in Agent Framework with ```Modal```

### Order of Play

1. **Modal.com and SpecialistAgent**
2. **RAG, FrontierAgent, Ensemble Agent**
3. **ScannerAgent, MessengerAgent**
4. **AutonomousPlannerAgent and DealAgentFramework**
5. **The Price Is Right Finale**

### ☁️ What is Modal?

**Modal** is a **serverless cloud platform designed for AI and compute-intensive applications**. It lets you run Python code, AI models, GPU workloads, training jobs, batch processing, and isolated environments **in the cloud without managing servers, Kubernetes, or traditional infrastructure**.

### 🧠 Simple way to remember it

> **Modal = Your Python code + Cloud GPUs/CPUs + Containers + Autoscaling + APIs**

Instead of buying or configuring a GPU server, you write Python and tell Modal what resources you need. Modal packages your code into a container and executes it in the cloud. It can automatically scale the number of containers based on demand.

In [ ]:
# Installing Libraries

# pip install --upgrade modal

In [ ]:
# Importing Libraries

import os
import locale
import modal
from agents.preprocessor import Preprocessor

In [ ]:
# To check that your computer can output special characters, make sure this outputs UTF-8 
# Should print 'UTF-8'

print(locale.getpreferredencoding()) 

In [ ]:
# if its not using the UTF-8 use this below code to set it
# os.environ["PYTHONIOENCODING"] = "utf-8"

### Setting up the modal tokens

### IMPORTANT - please do read and follow these instructions!

First please visit: https://modal.com

And sign up for an account. Then click the Avatar menu on the top right and select "Settings"

Then click "API Tokens" in the left sidebar, then click "New Token".

You will be given something like this to run:

```modal token set --token-id ak-somethinghere --token-secret as-somethinghere```

It might be totally fine to simply add the 2 keys directly to your .env file:

MODAL_TOKEN_ID=ak-...

MODAL_TOKEN_SECRET=as-...

In [ ]:
from hello import app, hello, hello_europe

In [ ]:
with app.run():
    reply = hello.local()
reply

In [ ]:
# the func.local will run the code locally and func.remote() will run this same code in modal remotely

with app.run():
    reply = hello.remote()
reply

If you look in hello.py, I've added a simple function hello_europe

That uses the decorator:
@app.function(image=image, region="eu")

See the result below! More region specific settings are [here](https://modal.com/docs/guide/region-selection)

Note that it does consume marginally more credits to specify a region.

In [ ]:
with app.run():
    reply = hello_europe.remote()
reply

#### We need to set your HuggingFace Token as a secret in Modal

**Super important - please read - this confuses a lot of people!**

Secrets in Modal are given a **name** that describes the secret.  
Then the secret itself has a KEY and a VALUE.  
We will be setting up a secret with:  

Name: huggingface-secret  
Key: HF_TOKEN  
Value: hf_...  

##### The bulletproof recipe:

1. Go to modal.com, sign in and go to your dashboard  
2. Click on Secrets in the nav bar  
3. Create new secret, click on Hugging Face, this new secret needs to be called **huggingface-secret** because that's how we refer to it in the code  
4. Fill in your key as HF_TOKEN and the value as your actual token hf_...  
5. Click done

#### And now back to business: time to work with Llama

In [ ]:
# This import may give a deprecation warning about adding local Python modules to the Image
# That warning can be safely ignored. You may get the same warning in other places, too..

from llama import app, generate

In [ ]:
with modal.enable_output():
    with app.run():
        results = generate.remote("Never gonna give you up, never gonna")
results

In [ ]:
from pricer_ephemeral import app, price, price_large

In [ ]:
with modal.enable_output():
    with app.run():
        result = price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
result

In [ ]:
with modal.enable_output():
    with app.run():
        result = price_large.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
result

In [ ]:
# Mow we will Preprocess the text and send it as input since we trained our model with some preprocessed text
# We will use local ollama llama 3.2 model to do this

preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

In [ ]:
import re
text = re.sub(r"(?m)^#.*\n?", "", text)
print(text)

In [ ]:
# Now we will feed this processed text to our models in remote

with modal.enable_output():
    with app.run():
        reply = price.remote(text)

print(reply)

In [ ]:
with modal.enable_output():
    with app.run():
        reply = price_large.remote(text)

print(reply)

#### Transitioning From Ephemeral Apps to Deployed Apps

From a command line, ```modal deploy xxx``` will deploy your code as a Deployed App

This is how you could package your AI service behind an API to be used in a Production System.

You can also build REST endpoints easily, although we won't cover that as we'll be calling direct from Python.

In [ ]:
# You can also run "modal deploy -m pricer_service" in the Terminal

!modal deploy -m pricer_service

In [ ]:
# Now Calling the remote Model

Pricer = modal.Cls.from_name("pricer-service", "Pricer")

In [ ]:
Pricer

In [ ]:
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
reply = pricer.price.remote(text)
print(reply)

#### Optional: Keeping Modal warm

A way to improve the speed of the Modal pricer service

The first time you run this modal class, it might take as much as 10 minutes to build.

Subsequently it should be much faster.. 30 seconds if it needs to wake up, otherwise 2 seconds.

If you want it to always be 2 seconds, you can keep the container from going to sleep by editing this constant in pricer_service2.py:

```MIN_CONTAINERS = 0```

Make it 1 to keep a container alive.
But please note: this will eat up credits! Only do this if you are comfortable to have a process running continually.

Alternatively, you can run this code and it will stay warm for 20 mins rather than 2 mins.

Code to keep warm for 20 mins before cooling down:
```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)
```

Code to revert to keeping warm for only 2 mins
```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=120)
```

#### And now introducing our Agent class

By default this will preprocess using Llama3.2

In [ ]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
agent = SpecialistAgent()

In [ ]:
agent.price("iPhone 10")